# 07 — TF activity in CD4 Teff: decoupleR (CollecTRI) and pySCENIC (Fig 5f)
Option B: TF activity from curated regulons (CollecTRI, univariate linear model). Not the paper's method,
but independent confirmation. Option A: reduced pySCENIC (equal cells per patient, few runs).

## Step 13.1 — Setup: load Teff cells from patients

In [1]:
suppressPackageStartupMessages({
  library(Seurat); library(tidyverse); library(decoupleR)
})
proj <- "/mnt/d/project3_IFN-I"
set.seed(42)

cd4  <- readRDS(file.path(proj, "results/objects/05_cd4_clean.rds"))
teff <- subset(cd4, subset = cd4_subset == "Teff" & group %in% c("high_IRC", "low_IRC"))
rm(cd4); invisible(gc())
teff$group <- factor(teff$group, levels = c("high_IRC", "low_IRC"))
table(teff$sample)


  P1   P2   P3   P4   P5   P6   P7   P8 
 587  937 1336  418  327  751  914  420 

## Step 13.2 — Get CollecTRI regulons and score TF activity per cell

In [3]:
url <- paste0("https://omnipathdb.org/interactions?datasets=collectri&genesymbols=yes",
              "&fields=is_stimulation,is_inhibition&license=academic")
raw <- read_tsv(url, show_col_types = FALSE)
colnames(raw)

net <- raw |>
  transmute(source = source_genesymbol,
            target = target_genesymbol,
            mor    = if_else(is_inhibition == 1 & is_stimulation == 0, -1, 1)) |>
  distinct(source, target, .keep_all = TRUE)
cat("CollecTRI:", n_distinct(net$source), "TFs,", nrow(net), "interactions\n")
head(net)

expr  <- LayerData(teff, layer = "data")
keep  <- rownames(expr) %in% net$target & Matrix::rowMeans(expr > 0) >= 0.01
mat   <- as.matrix(expr[keep, ])
cat("Genes used:", nrow(mat), " | cells:", ncol(mat), "\n")

acts <- run_ulm(mat = mat, net = net, .source = "source", .target = "target",
                .mor = "mor", minsize = 5)
rm(expr, mat); invisible(gc())

[1] "source"                "target"                "source_genesymbol"    
 [4] "target_genesymbol"     "is_directed"           "is_stimulation"       
 [7] "is_inhibition"         "consensus_direction"   "consensus_stimulation"
[10] "consensus_inhibition"

CollecTRI: 1201 TFs, 62411 interactions


source,target,mor
<chr>,<chr>,<dbl>
JUN_JUND,JUN,1
FOS_JUNB,JUN,1
JUNB,JUN,1
FOS_JUND,JUN,1
WT1,NR0B1,1
FOSB_JUN,JUN,1


Genes used: 3633  | cells: 5690 


## Step 13.3 — Compare TF activity, low vs high IRC (patient level)

In [4]:
tf_pt <- acts |>
  filter(statistic == "ulm") |>
  left_join(teff@meta.data |> rownames_to_column("condition") |> dplyr::select(condition, sample, group),
            by = "condition") |>
  group_by(source, sample, group) |>
  summarise(act = mean(score), .groups = "drop")

tf_cmp <- tf_pt |>
  group_by(source) |>
  summarise(high_mean = mean(act[group == "high_IRC"]),
            low_mean  = mean(act[group == "low_IRC"]),
            diff_low_minus_high = low_mean - high_mean,
            complete_separation = min(act[group == "low_IRC"]) > max(act[group == "high_IRC"]) |
                                  max(act[group == "low_IRC"]) < min(act[group == "high_IRC"]),
            p = suppressWarnings(wilcox.test(act ~ group, exact = TRUE)$p.value),
            .groups = "drop") |>
  arrange(desc(diff_low_minus_high))
write_csv(tf_cmp, file.path(proj, "results/tables/07_decoupler_tf_teff.csv"))

cat("Top 10 more active in LOW IRC:\n")
tf_cmp |> slice_head(n = 10) |> mutate(across(where(is.numeric), ~ signif(.x, 3)))
cat("Top 10 more active in HIGH IRC:\n")
tf_cmp |> slice_tail(n = 10) |> arrange(diff_low_minus_high) |> mutate(across(where(is.numeric), ~ signif(.x, 3)))

Top 10 more active in LOW IRC:


source,high_mean,low_mean,diff_low_minus_high,complete_separation,p
<chr>,<dbl>,<dbl>,<dbl>,<lgl>,<dbl>
HIVEP2,2.7300,4.620,1.890,TRUE,0.0286
MYC,0.9370,2.780,1.840,TRUE,0.0286
ZBTB4,2.0800,3.150,1.070,TRUE,0.0286
CIITA,0.8250,1.880,1.060,TRUE,0.0286
HLF,-1.0000,-0.256,0.748,FALSE,0.0571
ZBED1,2.1300,2.850,0.721,TRUE,0.0286
RFX1,-0.1590,0.534,0.693,TRUE,0.0286
HMGA2,-0.0783,0.535,0.614,FALSE,0.0571
TBXT,-0.2540,0.303,0.557,TRUE,0.0286


Top 10 more active in HIGH IRC:


source,high_mean,low_mean,diff_low_minus_high,complete_separation,p
<chr>,<dbl>,<dbl>,<dbl>,<lgl>,<dbl>
TCF7,2.03,1.120,-0.909,FALSE,0.2000
SATB2,2.61,1.740,-0.879,FALSE,0.0571
KAT6A,1.96,1.190,-0.771,FALSE,0.1140
CTBP2,1.67,1.000,-0.671,FALSE,0.1140
HOXC6,2.53,1.940,-0.590,FALSE,0.1140
BACH2,1.38,0.792,-0.588,FALSE,0.1140
MYBL1,1.78,1.200,-0.578,FALSE,0.2000
KLF8,2.81,2.240,-0.571,FALSE,0.1140
MYF5,2.12,1.560,-0.561,FALSE,0.0571


## Step 13.4 — Compare with the paper's top regulons (Fig 5f)

In [5]:
paper_5f <- tibble(
  source = c("THAP11", "FOS", "NELFE", "FOSB", "JUN", "JUND", "HMGB2", "IRF1", "JUNB", "TAF7",
             "ETV6", "HLF", "RORA", "STAT1", "CUX1", "FLI1", "ZMIZ1", "RUNX1", "MTA3", "MAF"),
  paper = rep(c("low IRC", "high IRC"), each = 10))

fig5f_check <- paper_5f |>
  left_join(tf_cmp, by = "source") |>
  mutate(ours = case_when(is.na(diff_low_minus_high) ~ "not in CollecTRI",
                          diff_low_minus_high > 0 ~ "low IRC", TRUE ~ "high IRC"),
         agree = paper == ours,
         across(where(is.numeric), ~ signif(.x, 3)))
fig5f_check

source,paper,high_mean,low_mean,diff_low_minus_high,complete_separation,p,ours,agree
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<lgl>,<dbl>,<chr>,<lgl>
THAP11,low IRC,NA,NA,NA,NA,NA,not in CollecTRI,FALSE
FOS,low IRC,0.5430,1.03000,0.4820,TRUE,0.0286,low IRC,TRUE
NELFE,low IRC,NA,NA,NA,NA,NA,not in CollecTRI,FALSE
FOSB,low IRC,-0.2320,-0.27200,-0.0404,FALSE,0.4860,high IRC,FALSE
JUN,low IRC,0.5700,0.92900,0.3600,TRUE,0.0286,low IRC,TRUE
JUND,low IRC,0.7670,0.92400,0.1570,FALSE,0.3430,low IRC,TRUE
HMGB2,low IRC,NA,NA,NA,NA,NA,not in CollecTRI,FALSE
IRF1,low IRC,-0.0819,0.17700,0.2590,FALSE,0.2000,low IRC,TRUE
JUNB,low IRC,1.2400,1.28000,0.0442,FALSE,0.6860,low IRC,TRUE


### Fig 5f — TF activity (decoupleR ULM, CollecTRI via OmniPath REST; patient-level)
- CollecTRI: 1,201 regulators (incl. AP-1 complexes), 62,411 interactions; 3,633 genes used.
- Low IRC top: MYC (MYC targets, consistent with translation/OXPHOS theme), CIITA and RFX1
  (MHC-II program; with CD74, B2M, HLA-C -> antigen-presentation program; new observation).
- High IRC top: FLI1, KAT6A, CTBP2, BACH2, TCF7 (none complete separation).
- Implausible TFs (TBXT, MYF5, HOXC6) -> generic-regulon artefacts.
- Paper's 20 regulons: 5 not in CollecTRI; 10/15 agree (low 5/6, high 5/9).
  Agree: FOS, JUN (complete sep.), JUND, IRF1, JUNB; MTA3 (complete sep.), FLI1, RUNX1, CUX1, ETV6.
  Disagree: MAF (complete sep., opposite), STAT1, HLF, RORA, FOSB.
- Disagreements are mostly the paper's smallest SCENIC regulons (MAF 9, RORA 9 genes);
  large regulons (225–1,509 genes) all agree.
- STAT1 opposite because CollecTRI includes canonical ISGs (IRF1, MHC) that paper's regulon excluded.
- Tfh claim (MAF) weakest-supported paper claim in our reanalysis.